In [48]:
import pandas as pd
import numpy as np
from sklearn.utils import resample

# Load your dataset
df = pd.read_csv('brain_stroke_encoded.csv')

# Separate the two classes
stroke_df = df[df['stroke'] == 1]  # Stroke cases (minority)
non_stroke_df = df[df['stroke'] == 0]  # Non-stroke cases (majority)

print(f"Original counts: Stroke={len(stroke_df)}, Non-Stroke={len(non_stroke_df)}")
print(f"Original ratio: Stroke={len(stroke_df)/len(df)*100:.2f}%")

# Determine the desired counts for 70% stroke, 30% non-stroke
# Let's create a balanced dataset with same total size (3750) or larger
total_samples = 3750  # You can adjust this
stroke_count_desired = int(0.7 * total_samples)  # 70% stroke
non_stroke_count_desired = int(0.3 * total_samples)  # 30% non-stroke

print(f"\nTarget counts for {total_samples} total samples:")
print(f"Stroke needed: {stroke_count_desired} (70%)")
print(f"Non-stroke needed: {non_stroke_count_desired} (30%)")

# Method 1: Oversample Stroke cases + Undersample Non-Stroke
# ---------------------------------------------------------

# 1. Oversample the minority class (stroke) with replacement
stroke_oversampled = resample(stroke_df,
                              replace=True,  # Sample with replacement
                              n_samples=stroke_count_desired,
                              random_state=42)

# 2. Undersample the majority class (non-stroke)
non_stroke_undersampled = resample(non_stroke_df,
                                   replace=False,  # Sample without replacement
                                   n_samples=non_stroke_count_desired,
                                   random_state=42)

# Combine the resampled classes
balanced_df = pd.concat([stroke_oversampled, non_stroke_undersampled])

# Shuffle the dataset
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nAfter balancing:")
print(f"Stroke: {len(balanced_df[balanced_df['stroke']==1])} ({len(balanced_df[balanced_df['stroke']==1])/len(balanced_df)*100:.2f}%)")
print(f"Non-Stroke: {len(balanced_df[balanced_df['stroke']==0])} ({len(balanced_df[balanced_df['stroke']==0])/len(balanced_df)*100:.2f}%)")
print(f"Total samples: {len(balanced_df)}")

# Save to a new CSV file
balanced_df.to_csv('brain_stroke_70_percent_stroke.csv', index=False)

Original counts: Stroke=250, Non-Stroke=4731
Original ratio: Stroke=5.00%

Target counts for 3750 total samples:
Stroke needed: 2625 (70%)
Non-stroke needed: 1125 (30%)

After balancing:
Stroke: 2625 (70.00%)
Non-Stroke: 1125 (30.00%)
Total samples: 3750


In [ ]:
# 1. CORRECT IMPORTS AND SETUP
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
# KEY: Use ImbPipeline to safely embed SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# 2. LOAD *ORIGINAL* DATA
# Ensure this CSV has the original, imbalanced class distribution (e.g., ~5% stroke)
print("Loading ORIGINAL dataset...")
df_original = pd.read_csv(r"C:\Users\sibs2\african-neurohealth-dashboard\brain_stroke_70_percent_stroke.csv")
print(f"Original class distribution:\n{df_original['stroke'].value_counts(normalize=True)}")

# 3. CREATE A PROPER, LEAK-PROOF TRAIN/TEST SPLIT
X = df_original.drop('stroke', axis=1)
y = df_original['stroke']
# Split ONCE. The test set is touched only at the very end.
X_train_full, X_test_final, y_train_full, y_test_final = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nFinal Test Set (held out): {X_test_final.shape[0]} samples")

# 4. DEFINE PREPROCESSING
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                           ('scale', StandardScaler())]), numeric_features),
        ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                           ('encode', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
    ])

# 5. BUILD A SINGLE, CORRECT PIPELINE WITH CROSS-VALIDATION
# This pipeline ensures SMOTE is applied ONLY during each training fold.
model_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42, sampling_strategy='auto')), # Adjust strategy if needed
    ('classifier', RandomForestClassifier(class_weight='balanced', random_state=42))
])

# 5a. (Optional) Tune hyperparameters using GridSearchCV on X_train_full
# 5b. Get cross-validated predictions on the training set for unbiased evaluation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_train_pred_proba = cross_val_predict(model_pipeline, X_train_full, y_train_full,
                                       cv=cv, method='predict_proba', n_jobs=-1)[:, 1]
print(f"\nCross-validated Training AUC: {roc_auc_score(y_train_full, y_train_pred_proba):.4f}")

# 6. FINAL TRAINING AND TEST EVALUATION (ON TRUE UNSEEN DATA)
# Train the final model on the entire training set
model_pipeline.fit(X_train_full, y_train_full)
# Predict on the FINAL, untouched test set
y_test_pred = model_pipeline.predict(X_test_final)
y_test_pred_proba = model_pipeline.predict_proba(X_test_final)[:, 1]

test_auc = roc_auc_score(y_test_final, y_test_pred_proba)
print("=" * 60)
print(f"✅ FINAL TEST on Unseen Data (Valid Metric)")
print(f"   Test Set AUC: {test_auc:.4f}")
print(f"   Test Set Classification Report:")
print(classification_report(y_test_final, y_test_pred))
print("=" * 60)



Loading ORIGINAL dataset...
Original class distribution:
stroke
1.0    0.7
0.0    0.3
Name: proportion, dtype: float64

Final Test Set (held out): 750 samples

Cross-validated Training AUC: 0.9998
✅ FINAL TEST on Unseen Data (Valid Metric)
   Test Set AUC: 1.0000
   Test Set Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      0.94      0.97       225
         1.0       0.98      1.00      0.99       525

    accuracy                           0.98       750
   macro avg       0.99      0.97      0.98       750
weighted avg       0.98      0.98      0.98       750



In [6]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ====== IMPORT ALL NECESSARY LIBRARIES ======
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report, 
    precision_score, recall_score, f1_score, confusion_matrix, 
    roc_curve, auc
)
from sklearn.pipeline import make_pipeline
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import label_binarize
import joblib
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, label_binarize
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    roc_auc_score, confusion_matrix, brier_score_loss, roc_curve, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score, classification_report, auc
)
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, HistGradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.multiclass import OneVsRestClassifier
from sklearn.datasets import make_classification
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## ====== LOAD YOUR DATASET ======
print("="*80)
print("LOADING DATASET WITH SMART PREPROCESSING")
print("="*80)

df = pd.read_csv(r"C:\Users\sibs2\african-neurohealth-dashboard\brain_stroke_70_percent_stroke.csv")

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst 3 rows:")
print(df.head(3))

# ====== INITIAL DATA EXPLORATION ======
print("\n" + "="*80)
print("INITIAL DATA EXPLORATION")
print("="*80)

# Check target column
if 'stroke' not in df.columns:
    raise ValueError("'stroke' column not found in dataset!")

print(f"Target column: 'stroke'")
print(f"Target unique values: {df['stroke'].unique()}")
print(f"Target distribution:")
print(df['stroke'].value_counts(dropna=False))
print(f"Target NaN values: {df['stroke'].isna().sum()}")

# ====== CLEAN TARGET VARIABLE ======
print("\n" + "="*80)
print("CLEANING TARGET VARIABLE")
print("="*80)

# Remove rows where target is NaN
df_clean = df.dropna(subset=['stroke']).copy()
print(f"Removed {len(df) - len(df_clean)} rows with NaN in target")

# Convert target to binary (0/1)
df_clean['stroke'] = df_clean['stroke'].astype(int)
print(f"\nTarget after cleaning:")
print(df_clean['stroke'].value_counts())
print(f"Stroke prevalence: {df_clean['stroke'].mean():.2%}")

# Check if we have enough stroke cases
stroke_count = df_clean['stroke'].sum()
if stroke_count < 100:
    print(f"⚠️ WARNING: Only {stroke_count} stroke cases - very imbalanced dataset!")
else:
    print(f"✅ {stroke_count} stroke cases available")

# ====== IDENTIFY AND REMOVE USELESS FEATURES ======
print("\n" + "="*80)
print("IDENTIFYING USELESS FEATURES")
print("="*80)

# Get features (exclude target)
X = df_clean.drop(columns=['stroke'])

# Check for columns with 100% missing values
missing_percentage = (X.isnull().sum() / len(X)) * 100
completely_missing = missing_percentage[missing_percentage == 100].index.tolist()

print(f"Columns with 100% missing values ({len(completely_missing)}):")
for col in completely_missing:
    print(f"  - {col}")

# Remove completely missing columns
if completely_missing:
    X = X.drop(columns=completely_missing)
    print(f"\nRemoved {len(completely_missing)} completely empty columns")

# Check for columns with very high missing rate (>50%)
high_missing = missing_percentage[(missing_percentage > 50) & (missing_percentage < 100)].index.tolist()
print(f"\nColumns with >50% missing values ({len(high_missing)}):")
for col in high_missing:
    print(f"  - {col}: {missing_percentage[col]:.1f}% missing")

# For this analysis, we'll also remove high-missing columns
if high_missing:
    X = X.drop(columns=high_missing)
    print(f"Removed {len(high_missing)} columns with >50% missing values")

# Check for constant columns (no variance)
constant_cols = []
for col in X.columns:
    if X[col].nunique() == 1:
        constant_cols.append(col)

if constant_cols:
    print(f"\nConstant columns ({len(constant_cols)}):")
    for col in constant_cols:
        print(f"  - {col}: value = {X[col].iloc[0]}")
    X = X.drop(columns=constant_cols)
    print(f"Removed {len(constant_cols)} constant columns")

print(f"\nFeatures after removing useless columns: {X.shape}")

# ====== HANDLE CATEGORICAL FEATURES ======
print("\n" + "="*80)
print("HANDLING CATEGORICAL FEATURES")
print("="*80)

categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
if categorical_cols:
    print(f"Found {len(categorical_cols)} categorical columns:")
    print(categorical_cols)
    
    # Process each categorical column
    for col in categorical_cols:
        # Fill missing with mode
        mode_val = X[col].mode()[0] if not X[col].mode().empty else 'Missing'
        X[col] = X[col].fillna(mode_val)
        
        # Convert to string and encode
        X[col] = X[col].astype(str)
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])
        
        print(f"  Encoded '{col}' with {len(le.classes_)} unique values")
else:
    print("No categorical columns found")

# ====== HANDLE NUMERIC FEATURES ======
print("\n" + "="*80)
print("HANDLING NUMERIC FEATURES")
print("="*80)

# Convert all remaining columns to numeric
for col in X.columns:
    if X[col].dtype not in ['int64', 'float64']:
        X[col] = pd.to_numeric(X[col], errors='coerce')

# Now handle missing values in numeric columns
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(f"Processing {len(numeric_cols)} numeric columns...")

# Check missing values
missing_counts = X[numeric_cols].isnull().sum()
missing_cols = missing_counts[missing_counts > 0].index.tolist()

if missing_cols:
    print(f"\nColumns with missing values after conversion:")
    for col in missing_cols[:10]:  # Show first 10
        print(f"  {col}: {missing_counts[col]} missing ({missing_counts[col]/len(X)*100:.1f}%)")
    
    # Impute with median
    for col in missing_cols:
        if X[col].notna().sum() > 0:  # Only if we have some non-NaN values
            median_val = X[col].median()
            X[col] = X[col].fillna(median_val)
            print(f"  Filled '{col}' with median: {median_val:.2f}")
        else:
            # If all values are NaN after conversion, drop the column
            print(f"  Dropping '{col}' - all values are NaN")
            X = X.drop(columns=[col])

print(f"\nFinal feature matrix shape: {X.shape}")

# Remove any rows that still have NaN values (should be very few)
rows_before = len(X)
X = X.dropna()
rows_after = len(X)
if rows_before > rows_after:
    print(f"Dropped {rows_before - rows_after} rows with remaining NaN values")

# Update target to match features
y = df_clean['stroke'].iloc[X.index]

print(f"\nFinal dataset:")
print(f"  Samples: {len(X)}")
print(f"  Features: {len(X.columns)}")
print(f"  Stroke cases: {y.sum()} ({y.mean():.2%})")

# ====== FEATURE SELECTION (Remove low variance features) ======
print("\n" + "="*80)
print("FEATURE SELECTION")
print("="*80)

# Calculate variance for each feature
variances = X.var()
low_variance_threshold = 0.01
low_variance_features = variances[variances < low_variance_threshold].index.tolist()

if low_variance_features:
    print(f"Found {len(low_variance_features)} features with variance < {low_variance_threshold}:")
    for col in low_variance_features[:10]:  # Show first 10
        print(f"  {col}: variance = {variances[col]:.6f}")
    
    X = X.drop(columns=low_variance_features)
    print(f"Removed {len(low_variance_features)} low-variance features")

print(f"\nFeatures after selection: {X.shape}")

# ====== SCALE FEATURES ======
print("\n" + "="*80)
print("SCALING FEATURES")
print("="*80)

# Keep original feature names
feature_names = X.columns.tolist()

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X = pd.DataFrame(X_scaled, columns=feature_names)

print(f"Features scaled. Mean ~0, Std ~1")
print(f"Feature ranges: min={X.min().min():.2f}, max={X.max().max():.2f}")

# ====== TRAIN-TEST SPLIT (80/20) ======
print("\n" + "="*80)
print("TRAIN-TEST SPLIT (80/20 for Random Forest)")
print("="*80)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Create aliases for different analysis sections
X_train_rf = X_train.copy()
X_test_rf = X_test.copy()
y_train_rf = y_train.copy()
y_test_rf = y_test.copy()

print(f"Training set: {X_train_rf.shape}")
print(f"Test set: {X_test_rf.shape}")
print(f"\nTraining class distribution:")
print(f"  No Stroke (0): {(y_train_rf == 0).sum()} ({(y_train_rf == 0).mean()*100:.1f}%)")
print(f"  Stroke (1): {(y_train_rf == 1).sum()} ({(y_train_rf == 1).mean()*100:.1f}%)")

# ====== TRAIN-TEST SPLIT (70/30) for Multi-Classifier Comparison ======
print("\n" + "="*80)
print("TRAIN-TEST SPLIT (70/30 for Multi-Classifier Comparison)")
print("="*80)

X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training set: {X_train_multi.shape}")
print(f"Test set: {X_test_multi.shape}")
print(f"\nTraining class distribution:")
print(f"  No Stroke (0): {(y_train_multi == 0).sum()} ({(y_train_multi == 0).mean()*100:.1f}%)")
print(f"  Stroke (1): {(y_train_multi == 1).sum()} ({(y_train_multi == 1).mean()*100:.1f}%)")

# ====== TRAIN SIMPLE BUT ROBUST MODEL ======
print("\n" + "="*80)
print("TRAINING MODEL (Handling Class Imbalance)")
print("="*80)

# With severe class imbalance, we need to handle it carefully
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Train model with class weighting
rf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=8,  # Limit depth to prevent overfitting
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'  # Automatically adjust for class imbalance
)

print("Training Random Forest with class weighting...")
rf_model.fit(X_train, y_train)
print("✅ Model trained")

# Cross-validation to check stability
cv_scores = cross_val_score(rf_model, X, y, cv=5, scoring='roc_auc')
print(f"\nCross-validation AUC scores (5-fold):")
print(f"  Scores: {cv_scores}")
print(f"  Mean: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")

# ====== EVALUATE MODEL ======
print("\n" + "="*80)
print("MODEL EVALUATION")
print("="*80)

y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)

try:
    roc_auc_val = roc_auc_score(y_test, y_pred_proba)
    print(f"ROC AUC: {roc_auc_val:.4f}")
except Exception as e:
    print(f"Could not calculate AUC: {e}")
    roc_auc_val = 0.5

print(f"Accuracy: {accuracy:.4f}")

# Performance interpretation
print(f"\n📊 Performance Assessment:")
if roc_auc_val < 0.6:
    print(f"  ⚠️  Poor discrimination (AUC < 0.6)")
elif roc_auc_val < 0.7:
    print(f"  ⚠️  Acceptable discrimination")
elif roc_auc_val < 0.8:
    print(f"  ✅ Good discrimination")
elif roc_auc_val < 0.9:
    print(f"  🎉 Very good discrimination")
else:
    print(f"  🏆 Excellent discrimination")

print("\n📈 Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Stroke', 'Stroke']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Stroke', 'Stroke'],
            yticklabels=['No Stroke', 'Stroke'])
plt.title('Confusion Matrix - Stroke Prediction', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix_final.png', dpi=300, bbox_inches='tight')
plt.show()

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
plt.figure(figsize=(10, 8))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc_val:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve - Stroke Prediction Model', fontsize=14, fontweight='bold')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curve_final.png', dpi=300, bbox_inches='tight')
plt.show()

# ====== FEATURE IMPORTANCE ======
print("\n" + "="*80)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*80)

# Get feature importance from Random Forest
feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 20 Features by RandomForest Importance:")
print(feature_importance.head(20).to_string(index=False))

# Plot feature importance
plt.figure(figsize=(12, 8))
top_n = min(20, len(feature_importance))
top_features = feature_importance.head(top_n)

plt.barh(range(top_n), top_features['Importance'], color='steelblue')
plt.yticks(range(top_n), top_features['Feature'], fontsize=10)
plt.xlabel('Feature Importance', fontsize=12)
plt.title(f'Top {top_n} Features for Stroke Prediction', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()

# Add value labels
for i, val in enumerate(top_features['Importance']):
    plt.text(val + 0.001, i, f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('feature_importance_final.png', dpi=300, bbox_inches='tight')
plt.show()

# ===========================================
# PART 1: RANDOM FOREST WITH SHAP (80/20 SPLIT)
# ===========================================
print("\n" + "="*80)
print("PART 1: RANDOM FOREST WITH SHAP ANALYSIS")
print("="*80)

# ====== TRAIN RANDOM FOREST ======
print("\n=== Training Random Forest ===")
stroke_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)

stroke_model.fit(X_train_rf, y_train_rf)

# ====== EVALUATE ======
y_pred_rf = stroke_model.predict(X_test_rf)
y_proba_rf = stroke_model.predict_proba(X_test_rf)[:, 1]

print("\n=== Performance ===")
print(f"Accuracy: {accuracy_score(y_test_rf, y_pred_rf):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test_rf, y_proba_rf):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_rf, y_pred_rf))

# ====== SAVE MODEL ======
joblib.dump(stroke_model, "stroke_REAL_model.pkl")
joblib.dump(stroke_model, "stroke_pipeline.joblib")
print("\n✅ STROKE model saved as stroke_REAL_model.pkl")

# ====== CALCULATE SHAP VALUES ======
print("\n=== Calculating SHAP Values ===")

# Initialize explainer and calculate SHAP
explainer = shap.TreeExplainer(stroke_model)

sample_size = min(100, len(X_test_rf))
X_sample = X_test_rf.iloc[:sample_size]
print(f"Using {sample_size} samples for SHAP calculation")

# Calculate SHAP values
shap_values = explainer.shap_values(X_sample)

# ====== PROCESS SHAP VALUES ======
# Handle different SHAP output formats
if isinstance(shap_values, list):
    # For binary classification: shap_values[0] = class 0, shap_values[1] = class 1
    if len(shap_values) == 2:
        shap_values_stroke = shap_values[1]  # Class 1 (Stroke)
    else:
        shap_values_stroke = shap_values[0]  # Fallback
elif len(shap_values.shape) == 3:
    # 3D array: (samples, features, classes)
    shap_values_stroke = shap_values[:, :, 1]  # Class 1 (Stroke)
else:
    # 2D array: (samples, features) - single class
    shap_values_stroke = shap_values

# Ensure we have the correct feature names
if hasattr(X_sample, 'columns'):
    feature_names = X_sample.columns.tolist()
elif hasattr(X_test_rf, 'columns'):
    feature_names = X_test_rf.columns.tolist()
else:
    feature_names = [f"Feature_{i}" for i in range(shap_values_stroke.shape[1])]

# Calculate SHAP statistics
mean_abs_shap = np.abs(shap_values_stroke).mean(axis=0)
mean_shap = shap_values_stroke.mean(axis=0)

# Create SHAP DataFrame
shap_df = pd.DataFrame({
    'Feature': feature_names,
    'Mean_Absolute_SHAP': mean_abs_shap,
    'Mean_SHAP': mean_shap
}).sort_values('Mean_Absolute_SHAP', ascending=False)

# Determine impact direction
shap_df['Impact'] = shap_df['Mean_SHAP'].apply(
    lambda x: 'Increases Risk' if x > 0 else 'Decreases Risk'
)

# Display top predictors
print("\n" + "="*80)
print("TOP 20 PREDICTORS - STROKE DATA MODEL")
print("="*80)
print(shap_df.head(20).to_string(index=False))

# Save SHAP values
shap_df.to_csv('stroke_REAL_shap_values.csv', index=False)
print("\n✅ Stroke SHAP values saved to stroke_REAL_shap_values.csv")

# ====== CLINICAL INSIGHTS ======
print("\n" + "="*80)
print("CLINICAL INSIGHTS")
print("="*80)

# Known stroke risk factors with more flexible matching
clinical_factors = {
    'age': 'Age',
    'hypertension': 'High Blood Pressure',
    'glucose': 'Blood Sugar/Diabetes',
    'heart': 'Heart Disease',
    'bmi': 'Body Mass Index (Obesity)',
    'smok': 'Smoking',
    'diabet': 'Diabetes',
    'cholesterol': 'Cholesterol',
    'alcohol': 'Alcohol Use',
    'physical': 'Physical Activity',
    'stress': 'Stress',
    'sleep': 'Sleep',
    'depress': 'Depression',
    'ptsd': 'PTSD',
    'stroke': 'Previous Stroke',
    'gender': 'Gender',
    'marital': 'Marital Status',
    'work': 'Work Type'
}

print("\nIdentifying clinical risk factors in top predictors:")
clinical_found = 0
clinical_insights = []

for i, row in shap_df.head(20).iterrows():
    feature_lower = row['Feature'].lower()
    matched = False
    
    # Check for clinical factor matches
    for factor_key, factor_name in clinical_factors.items():
        if factor_key in feature_lower:
            arrow = "↑" if row['Impact'] == 'Increases Risk' else "↓"
            insight = {
                'rank': clinical_found + 1,
                'feature': row['Feature'],
                'clinical_factor': factor_name,
                'direction': arrow,
                'shap_value': row['Mean_SHAP'],
                'abs_shap': row['Mean_Absolute_SHAP']
            }
            clinical_insights.append(insight)
            clinical_found += 1
            matched = True
            break
    
    # If no clinical factor matched, track as non-clinical
    if not matched:
        arrow = "↑" if row['Impact'] == 'Increases Risk' else "↓"
        insight = {
            'rank': i + 1,
            'feature': row['Feature'],
            'clinical_factor': 'Other/Non-clinical',
            'direction': arrow,
            'shap_value': row['Mean_SHAP'],
            'abs_shap': row['Mean_Absolute_SHAP']
        }
        clinical_insights.append(insight)

# Display clinical insights
if clinical_found > 0:
    print(f"\nFound {clinical_found} clinical risk factors in top 20 predictors:")
    print("-" * 80)
    print(f"{'#':<3} {'Feature':<25} {'Clinical Factor':<25} {'Impact':<25} {'SHAP Value':<25}")
    print("-" * 80)
    
    for insight in sorted(clinical_insights, key=lambda x: x['abs_shap'], reverse=True)[:10]:
        if insight['clinical_factor'] != 'Other/Non-clinical':
            print(f"{insight['rank']:<3} {insight['feature']:<25} {insight['clinical_factor']:<25} "
                  f"{insight['direction']:<12} {insight['shap_value']:>9.4f}")
else:
    print("No known clinical risk factors found in top predictors")
    print("\nTop predictors are:")
    for i, row in shap_df.head(10).iterrows():
        arrow = "↑" if row['Impact'] == 'Increases Risk' else "↓"
        print(f"  {i+1:2d}. {row['Feature']:25s} {arrow} (SHAP: {row['Mean_SHAP']:.4f})")

# Optional: Create summary report
clinical_summary = pd.DataFrame(clinical_insights)
clinical_summary.to_csv('clinical_insights_summary.csv', index=False)
print("\n✅ Clinical insights summary saved to clinical_insights_summary.csv")

# ===========================================
# PART 2: MULTI-CLASSIFIER COMPARISON (70/30 SPLIT)
# ===========================================
print("\n" + "="*80)
print("PART 2: MULTI-CLASSIFIER COMPARISON")
print("="*80)

# ====== DEFINE CLASSIFIERS ======
print("\n=== Defining Classifiers ===")
classifiers = [
    make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    make_pipeline(StandardScaler(), SVC(kernel="linear", probability=True, C=1.0, max_iter=2000, random_state=42)),
    GaussianProcessClassifier(kernel=1.0 * RBF(1.0), random_state=42),
    DecisionTreeClassifier(max_depth=6, random_state=42),
    RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42),
    AdaBoostClassifier(n_estimators=150, random_state=42),
    make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, C=1.0, solver="lbfgs", random_state=42)),
    GaussianNB()
]

names = [
    "Nearest Neighbors",
    "Linear SVM",
    "Gaussian Process",
    "Decision Tree",
    "Random Forest",
    "AdaBoost",
    "Logistic Regression",
    "Naive Bayes"
]

# ====== ACCURACY HELPER FUNCTION ======
def cal_accuracy(y_true, y_pred, model_name):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    print(f"\n📊 {model_name} Metrics:")
    print(f"   Accuracy : {accuracy:.4f}")
    print(f"   Precision: {precision:.4f}")
    print(f"   Recall   : {recall:.4f}")
    print(f"   F1-score : {f1:.4f}")
    print("\nClassification Report:\n", classification_report(y_true, y_pred))

# ====== TRAIN AND EVALUATE ALL CLASSIFIERS ======
print("\n=== Training and Evaluating All Classifiers ===")
results = []

for name, clf in zip(names, classifiers):
    print(f"\n{'='*50}")
    print(f"Training: {name}")
    print('='*50)
    
    # Train the classifier
    clf.fit(X_train_multi, y_train_multi)
    
    # Make predictions
    y_pred = clf.predict(X_test_multi)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test_multi, y_pred) * 100
    ppv = precision_score(y_test_multi, y_pred, average="weighted", zero_division=0) * 100
    sensitivity = recall_score(y_test_multi, y_pred, average="weighted", zero_division=0) * 100
    f1 = f1_score(y_test_multi, y_pred, average="weighted", zero_division=0) * 100
    
    # Specificity
    cm = confusion_matrix(y_test_multi, y_pred)
    specificity_list = []
    for i in range(len(cm)):
        tn = cm.sum() - (cm[i, :].sum() + cm[:, i].sum() - cm[i, i])
        fp = cm[:, i].sum() - cm[i, i]
        specificity_list.append(tn / (tn + fp) if (tn + fp) > 0 else 0)
    specificity = np.mean(specificity_list) * 100
    
    # NPV
    npv_list = []
    for i in range(len(cm)):
        tn = cm.sum() - (cm[i, :].sum() + cm[:, i].sum() - cm[i, i])
        fn = cm[i, :].sum() - cm[i, i]
        npv_list.append(tn / (tn + fn) if (tn + fn) > 0 else 0)
    npv = np.mean(npv_list) * 100
    
    # AUC
    try:
        if hasattr(clf, "predict_proba"):
            y_score = clf.predict_proba(X_test_multi)
            if y_score.shape[1] == 2:  # Binary classification
                roc_auc = roc_auc_score(y_test_multi, y_score[:, 1])
            else:
                roc_auc = roc_auc_score(y_test_multi, y_score, multi_class='ovr')
        else:
            y_score = clf.decision_function(X_test_multi)
            roc_auc = roc_auc_score(y_test_multi, y_score)
    except:
        roc_auc = 0.0
    
    # Store results
    results.append({
        "Algorithm": name,
        "Accuracy (%)": round(accuracy, 2),
        "PPV (%)": round(ppv, 2),
        "NPV (%)": round(npv, 2),
        "Sensitivity (%)": round(sensitivity, 2),
        "Specificity (%)": round(specificity, 2),
        "AUC": round(roc_auc, 4)
    })
    
    # Print detailed metrics
    cal_accuracy(y_test_multi, y_pred, name)

# ====== DISPLAY METRICS TABLE ======
print("\n" + "="*80)
print("CLASSIFIER METRICS SUMMARY")
print("="*80)

df_results = pd.DataFrame(results)
print("\n✅ Classifier Metrics Table:\n")
print(df_results.to_string(index=False))

# Save results to files
df_results.to_csv("stroke_classifier_metrics.csv", index=False)
df_results.to_excel("stroke_classifier_metrics.xlsx", index=False)
print("\n✅ Metrics saved to CSV and Excel files")

# ====== CREATE ROC CURVES ======
print("\n=== Generating ROC Curves ===")

# Binarize the labels for ROC
y_test_bin = label_binarize(y_test_multi, classes=[0, 1])

fig, axes = plt.subplots(4, 2, figsize=(16, 20))
axes = axes.flatten()

for idx, (name, clf) in enumerate(zip(names, classifiers)):
    ax = axes[idx]
    
    # Get predictions
    try:
        if hasattr(clf, "predict_proba"):
            y_score = clf.predict_proba(X_test_multi)[:, 1]
        else:
            y_score = clf.decision_function(X_test_multi)
    except:
        continue
    
    # Calculate ROC curve
    fpr, tpr, thresholds = roc_curve(y_test_multi, y_score)
    roc_auc = auc(fpr, tpr)
    
    # Calculate Youden's index for optimal threshold
    youden_index = np.argmax(tpr - fpr)
    optimal_sensitivity = tpr[youden_index]
    optimal_specificity = 1 - fpr[youden_index]
    
    # Plot ROC curve
    ax.plot(fpr, tpr, color='darkorange', lw=3, 
            label=f'AUC = {roc_auc:.3f}\nSens = {optimal_sensitivity:.2f}\nSpec = {optimal_specificity:.2f}')
    ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'{name} ROC Curve')
    ax.legend(loc="lower right")
    ax.grid(True, alpha=0.3)

# Hide empty subplots if any
for idx in range(len(classifiers), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('ROC Curves for Different Classifiers (Stroke Prediction)', fontsize=16, y=1.02)
plt.tight_layout()

# Save plots
plt.savefig("stroke_ROC_AUC_Sensitivity.png", dpi=300, bbox_inches="tight")
plt.savefig("stroke_ROC_AUC_Sensitivity.pdf", dpi=300, bbox_inches="tight")
plt.show()

print("✅ ROC, AUC, and Sensitivity saved as PNG + PDF")

# ====== CREATE SHAP SUMMARY PLOT ======
print("\n=== Creating SHAP Summary Plot ===")
try:
    plt.figure(figsize=(14, 8))
    shap.summary_plot(shap_values_stroke, X_sample, feature_names=X.columns.tolist(), show=False)
    plt.tight_layout()
    plt.savefig('stroke_REAL_shap_summary.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(shap_df.head(20).to_string(index=False))
    print("✅ SHAP plot saved as stroke_REAL_shap_summary.png")
except Exception as e:
    print(f"Could not create SHAP plot: {e}")

# ====== CHECK MODEL FEATURE IMPORTANCE ======
print("\n=== Random Forest Feature Importance ===")
rf_stroke_model = pd.DataFrame({
    'Feature': X.columns,
    'RF_Importance': stroke_model.feature_importances_
}).sort_values('RF_Importance', ascending=False)

print("Top 20 by RF importance:")
print(rf_stroke_model.head(20).to_string(index=False))

# Compare with SHAP
print("\n=== Comparing SHAP vs RF Importance (Top 20) ===")
for i in range(min(20, len(shap_df))):
    shap_pred = shap_df.iloc[i]['Feature']
    shap_val = shap_df.iloc[i]['Mean_Absolute_SHAP']

    rf_row = rf_stroke_model[rf_stroke_model['Feature'] == shap_pred]
    if not rf_row.empty:
        rf_val = rf_row.iloc[0]['RF_Importance']
        print(f"{i+1}. {shap_pred:30s} SHAP: {shap_val:.4f}, RF: {rf_val:.4f}")

# ====== SAVE COMPLETE FEATURE IMPORTANCE COMPARISON ======
comparison_df = pd.merge(
    shap_df[['Feature','Mean_Absolute_SHAP', 'Impact']],
    rf_stroke_model[['Feature', 'RF_Importance']],
    on='Feature',
    how='left'
)
comparison_df = comparison_df.sort_values('Mean_Absolute_SHAP', ascending=False)
comparison_df.to_csv('stroke_feature_importance_comparison.csv', index=False)
print("\n✅ Feature importance comparison saved to stroke_feature_importance_comparison.csv")

# ====== FINAL SUMMARY ======
print("\n" + "="*80)
print("ANALYSIS COMPLETE - SUMMARY")
print("="*80)
print(f"Dataset: {df.shape[0]} samples, {df.shape[1]} features")
print(f"Target distribution: {np.bincount(y)}")
print(f"\nRandom Forest Model (80/20 split):")
print(f"  Accuracy: {accuracy_score(y_test_rf, y_pred_rf):.4f}")
print(f"  ROC AUC: {roc_auc_score(y_test_rf, y_proba_rf):.4f}")
print(f"\nBest Classifier from Multi-Comparison (70/30 split):")
best_alg = df_results.loc[df_results['Accuracy (%)'].idxmax(), 'Algorithm']
best_acc = df_results['Accuracy (%)'].max()
print(f"  {best_alg}: {best_acc:.2f}% accuracy")
print(f"\nFiles Saved:")
print(f"  1. stroke_REAL_model.pkl - Trained Random Forest model")
print(f"  2. stroke_REAL_shap_values.csv - SHAP feature importance")
print(f"  3. stroke_classifier_metrics.csv - Multi-classifier results")
print(f"  4. stroke_feature_importance_comparison.csv - SHAP vs RF comparison")
print(f"  5. stroke_REAL_shap_summary.png - SHAP visualization")
print(f"  6. stroke_ROC_AUC_Sensitivity.png - ROC curves")


# ============================================================================
# 6. CONFUSION MATRIX (FIXED)
# ============================================================================
print("\n" + "="*80)
print("6. CONFUSION MATRIX")
print("="*80)

cm = confusion_matrix(y_test_multi, y_pred)
tn, fp, fn, tp = cm.ravel()

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

print(f"Confusion Matrix:")
print(f"  True Negatives: {tn}")
print(f"  False Positives: {fp}")
print(f"  False Negatives: {fn}")
print(f"  True Positives: {tp}")
print(f"\n  Sensitivity (Recall): {sensitivity:.4f}")
print(f"  Specificity: {specificity:.4f}")

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Stroke', 'Stroke'],
            yticklabels=['No Stroke', 'Stroke'])
plt.title('Confusion Matrix - Stroke Prediction\n(Test Set Evaluation)', 
          fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix_final.png', dpi=300, bbox_inches='tight')
plt.show()

import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams

# Set style for better visualizations
plt.style.use('seaborn-v0_8-whitegrid')
rcParams['figure.figsize'] = (12, 8)
rcParams['font.size'] = 11

# ====== CALCULATE SHAP VALUES ======
print("\n=== Calculating SHAP Values ===")

# Initialize explainer and calculate SHAP
explainer = shap.TreeExplainer(stroke_model)

sample_size = min(100, len(X_test_rf))
X_sample = X_test_rf.iloc[:sample_size]
print(f"Using {sample_size} samples for SHAP calculation")

# Calculate SHAP values
shap_values = explainer.shap_values(X_sample)

# ====== PROCESS SHAP VALUES ======
# Handle different SHAP output formats
if isinstance(shap_values, list):
    # For binary classification: shap_values[0] = class 0, shap_values[1] = class 1
    if len(shap_values) == 2:
        shap_values_stroke = shap_values[1]  # Class 1 (Stroke)
    else:
        shap_values_stroke = shap_values[0]  # Fallback
elif len(shap_values.shape) == 3:
    # 3D array: (samples, features, classes)
    shap_values_stroke = shap_values[:, :, 1]  # Class 1 (Stroke)
else:
    # 2D array: (samples, features) - single class
    shap_values_stroke = shap_values

# Ensure we have the correct feature names
if hasattr(X_sample, 'columns'):
    feature_names = X_sample.columns.tolist()
elif hasattr(X_test_rf, 'columns'):
    feature_names = X_test_rf.columns.tolist()
else:
    feature_names = [f"Feature_{i}" for i in range(shap_values_stroke.shape[1])]

# Calculate SHAP statistics
mean_abs_shap = np.abs(shap_values_stroke).mean(axis=0)
mean_shap = shap_values_stroke.mean(axis=0)

# Create SHAP DataFrame
shap_df = pd.DataFrame({
    'Feature': feature_names,
    'Mean_Absolute_SHAP': mean_abs_shap,
    'Mean_SHAP': mean_shap
}).sort_values('Mean_Absolute_SHAP', ascending=False)

# Determine impact direction
shap_df['Impact'] = shap_df['Mean_SHAP'].apply(
    lambda x: 'Increases Risk' if x > 0 else 'Decreases Risk'
)

# Display top predictors
print("\n" + "="*80)
print("TOP 20 PREDICTORS - STROKE DATA MODEL")
print("="*80)
print(shap_df.head(20).to_string(index=False))

# Save SHAP values
shap_df.to_csv('stroke_REAL_shap_values.csv', index=False)
print("\n✅ Stroke SHAP values saved to stroke_REAL_shap_values.csv")

# ====== SHAP VISUALIZATION PLOTS ======
print("\n" + "="*80)
print("GENERATING SHAP VISUALIZATIONS")
print("="*80)

# Create directory for plots if it doesn't exist
import os
if not os.path.exists('shap_plots'):
    os.makedirs('shap_plots')

# 1. BAR PLOT - All Features (Global Importance)
print("\n1. Generating SHAP Bar Plot (All Features)...")
plt.figure(figsize=(14, 10))
shap.summary_plot(shap_values_stroke, X_sample, plot_type="bar", 
                  show=False, max_display=len(feature_names))
plt.title("SHAP Feature Importance - All Features", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_plots/shap_bar_plot_all_features.png', dpi=300, bbox_inches='tight')
print("   ✅ Saved: shap_plots/shap_bar_plot_all_features.png")

# 2. BEESWARM PLOT - All Features
print("2. Generating SHAP Beeswarm Plot (All Features)...")
plt.figure(figsize=(16, 10))
shap.summary_plot(shap_values_stroke, X_sample, 
                  show=False, max_display=len(feature_names))
plt.title("SHAP Values Impact - All Features", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_plots/shap_beeswarm_all_features.png', dpi=300, bbox_inches='tight')
print("   ✅ Saved: shap_plots/shap_beeswarm_all_features.png")


# 4. CUSTOM SORTED BAR PLOT - All Features with Colors
print("4. Generating Custom Sorted SHAP Bar Plot...")
fig, ax = plt.subplots(figsize=(14, 10))

# Sort features by absolute SHAP value
sorted_idx = np.argsort(np.abs(shap_values_stroke).mean(0))[::-1]
sorted_features = [feature_names[i] for i in sorted_idx]
sorted_mean_shap = [mean_shap[i] for i in sorted_idx]

# Color by positive/negative impact
colors = ['red' if val > 0 else 'blue' for val in sorted_mean_shap]

y_pos = np.arange(len(sorted_features))
ax.barh(y_pos, sorted_mean_shap, align='center', color=colors, alpha=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(sorted_features)
ax.invert_yaxis()
ax.set_xlabel('Mean SHAP Value (Impact on Stroke Risk)', fontsize=12)
ax.set_title('All Features: SHAP Value Impact (Red = Increases Risk, Blue = Decreases Risk)', 
             fontsize=14, fontweight='bold')

# Add value labels
for i, v in enumerate(sorted_mean_shap):
    ax.text(v + (0.01 if v >= 0 else -0.01), i, 
            f'{v:.4f}', 
            color='black' if abs(v) > 0.01 else 'gray',
            va='center',
            fontsize=9)

plt.tight_layout()
plt.savefig('shap_plots/shap_custom_bar_all_features.png', dpi=300, bbox_inches='tight')
print("   ✅ Saved: shap_plots/shap_custom_bar_all_features.png")

# 5. VIOLIN PLOT - Distribution of SHAP values for all features
print("5. Generating SHAP Violin Plot (All Features)...")
fig, ax = plt.subplots(figsize=(16, 12))

# Create DataFrame for violin plot
shap_long = pd.DataFrame(shap_values_stroke, columns=feature_names)
shap_long_melted = shap_long.melt(var_name='Feature', value_name='SHAP Value')

# Get top features for better visualization
top_features = shap_df.head(30)['Feature'].tolist()
shap_long_melted_top = shap_long_melted[shap_long_melted['Feature'].isin(top_features)]

# Create violin plot
sns.violinplot(data=shap_long_melted_top, x='SHAP Value', y='Feature', 
               palette='viridis', ax=ax)
ax.axvline(x=0, color='black', linestyle='--', alpha=0.5)
ax.set_title('Distribution of SHAP Values - Top 30 Features', fontsize=16, fontweight='bold')
ax.set_xlabel('SHAP Value (Impact on Stroke Risk)', fontsize=12)
ax.set_ylabel('Features', fontsize=12)

plt.tight_layout()
plt.savefig('shap_plots/shap_violin_plot.png', dpi=300, bbox_inches='tight')
print("   ✅ Saved: shap_plots/shap_violin_plot.png")


# ====== CLINICAL INSIGHTS ======
print("\n" + "="*80)
print("CLINICAL INSIGHTS")
print("="*80)

# Known stroke risk factors with more flexible matching
clinical_factors = {
    'age': 'Age',
    'hypertension': 'High Blood Pressure',
    'glucose': 'Blood Sugar/Diabetes',
    'heart': 'Heart Disease',
    'bmi': 'Body Mass Index (Obesity)',
    'smok': 'Smoking',
    'diabet': 'Diabetes',
    'cholesterol': 'Cholesterol',
    'alcohol': 'Alcohol Use',
    'physical': 'Physical Activity',
    'stress': 'Stress',
    'sleep': 'Sleep',
    'depress': 'Depression',
    'ptsd': 'PTSD',
    'stroke': 'Previous Stroke',
    'gender': 'Gender',
    'marital': 'Marital Status',
    'work': 'Work Type',
    'blood': 'Blood Pressure',
    'sugar': 'Blood Sugar'
}

print("\nIdentifying clinical risk factors in ALL features:")
clinical_insights = []

# Analyze ALL features, not just top 20
for i, row in shap_df.iterrows():
    feature_lower = row['Feature'].lower()
    matched = False
    
    # Check for clinical factor matches
    for factor_key, factor_name in clinical_factors.items():
        if factor_key in feature_lower:
            arrow = "↑" if row['Impact'] == 'Increases Risk' else "↓"
            insight = {
                'feature': row['Feature'],
                'clinical_factor': factor_name,
                'impact': row['Impact'],
                'direction': arrow,
                'mean_shap': row['Mean_SHAP'],
                'abs_shap': row['Mean_Absolute_SHAP'],
                'rank': i + 1
            }
            clinical_insights.append(insight)
            matched = True
            break
    
    # If no clinical factor matched
    if not matched:
        arrow = "↑" if row['Impact'] == 'Increases Risk' else "↓"
        insight = {
            'feature': row['Feature'],
            'clinical_factor': 'Other/Non-clinical',
            'impact': row['Impact'],
            'direction': arrow,
            'mean_shap': row['Mean_SHAP'],
            'abs_shap': row['Mean_Absolute_SHAP'],
            'rank': i + 1
        }
        clinical_insights.append(insight)

# Convert to DataFrame and sort
clinical_df = pd.DataFrame(clinical_insights)
clinical_df_sorted = clinical_df.sort_values('abs_shap', ascending=False)

# Display clinical summary
clinical_count = len([x for x in clinical_df['clinical_factor'] if x != 'Other/Non-clinical'])
total_features = len(clinical_df)

print(f"\nClinical Analysis Summary:")
print(f"Total Features Analyzed: {total_features}")
print(f"Clinical Risk Factors Identified: {clinical_count}")
print(f"Non-clinical Features: {total_features - clinical_count}")

# Save clinical insights
clinical_df_sorted.to_csv('clinical_insights_all_features.csv', index=False)
print("\n✅ Clinical insights saved to clinical_insights_all_features.csv")

# Create summary visualization of clinical vs non-clinical factors
plt.figure(figsize=(10, 8))
clinical_df_sorted['is_clinical'] = clinical_df_sorted['clinical_factor'] != 'Other/Non-clinical'

# Plot clinical vs non-clinical SHAP importance
clinical_shap = clinical_df_sorted[clinical_df_sorted['is_clinical']]['abs_shap'].sum()
non_clinical_shap = clinical_df_sorted[~clinical_df_sorted['is_clinical']]['abs_shap'].sum()

labels = ['Clinical Factors', 'Non-Clinical Factors']
values = [clinical_shap, non_clinical_shap]
colors = ['#FF6B6B', '#4ECDC4']

plt.pie(values, labels=labels, colors=colors, autopct='%1.1f%%', 
        startangle=90, explode=(0.1, 0), shadow=True)
plt.title('SHAP Importance Distribution: Clinical vs Non-Clinical Factors', 
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_plots/clinical_vs_non_clinical_pie.png', dpi=300, bbox_inches='tight')
print("   ✅ Saved: shap_plots/clinical_vs_non_clinical_pie.png")

# Print top clinical factors
print("\n" + "="*80)
print("TOP 10 CLINICAL RISK FACTORS (by SHAP importance)")
print("="*80)

top_clinical = clinical_df_sorted[clinical_df_sorted['is_clinical']].head(10)
print(f"\n{'Rank':<5} {'Feature':<25} {'Clinical Factor':<25} {'Impact':<15} {'SHAP Value':>10}")
print("-" * 85)

for i, row in top_clinical.iterrows():
    print(f"{row['rank']:<5} {row['feature']:<25} {row['clinical_factor']:<25} "
          f"{row['impact']:<15} {row['mean_shap']:>10.4f}")

print("\n" + "="*80)
print("VISUALIZATION COMPLETE")
print("="*80)
print(f"\nAll SHAP plots saved to 'shap_plots/' directory:")
print("1. Bar plot (all features)")
print("2. Beeswarm plot (all features)")
print("3. Heatmap (top 20 features)")
print("4. Custom bar plot (all features with colors)")
print("5. Violin plot (distribution analysis)")
print("6. Individual waterfall plots")
print("7. Interactive force plots (HTML)")

plt.close('all')  # Close all figures to free memory

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ====== IMPORT ALL NECESSARY LIBRARIES ======
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report, 
    precision_score, recall_score, f1_score, confusion_matrix, 
    roc_curve, auc
)
from sklearn.pipeline import make_pipeline
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import label_binarize
import joblib
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, label_binarize
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    roc_auc_score, confusion_matrix, brier_score_loss, roc_curve, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score, classification_report, auc
)
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, HistGradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.multiclass import OneVsRestClassifier
from sklearn.datasets import make_classification
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.calibration import calibration_curve
from sklearn.metrics import (brier_score_loss, roc_auc_score, accuracy_score, 
                             precision_score, recall_score, f1_score, 
                             confusion_matrix, classification_report, 
                             roc_curve, auc)
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8-whitegrid')
rcParams['figure.figsize'] = (12, 8)
rcParams['font.size'] = 11
sns.set_palette("husl")

# ====== 10-FOLD CROSS-VALIDATION ======
print("\n" + "="*80)
print("10-FOLD CROSS-VALIDATION")
print("="*80)

# Prepare data for cross-validation - Use all available data
X_train_full = X.copy() if 'X' in locals() else X_train
y_train_full = y.copy() if 'y' in locals() else y_train

# Ensure they have matching indices
if not X_train_full.index.equals(y_train_full.index):
    # Reset indices to match
    X_train_full = X_train_full.reset_index(drop=True)
    y_train_full = y_train_full.reset_index(drop=True)

# Initialize stratified K-Fold
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Store CV results
cv_scores = {
    'roc_auc': [],
    'accuracy': [],
    'precision': [],
    'recall': [],
    'f1': [],
    'brier': []
}

# Store predictions for calibration as arrays (not lists)
y_true_cv = np.array([], dtype=int)
y_pred_proba_cv = np.array([])

# Cross-validation loop
print("\nPerforming 10-fold cross-validation...")
for fold, (train_idx, val_idx) in enumerate(cv.split(X_train_full, y_train_full), 1):
    X_train_cv = X_train_full.iloc[train_idx].reset_index(drop=True)
    X_val_cv = X_train_full.iloc[val_idx].reset_index(drop=True)
    y_train_cv = y_train_full.iloc[train_idx].reset_index(drop=True)
    y_val_cv = y_train_full.iloc[val_idx].reset_index(drop=True)
    
    # Train model on training fold
    model_cv = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced'
    )
    model_cv.fit(X_train_cv, y_train_cv)
    
    # Predict on validation fold
    y_pred = model_cv.predict(X_val_cv)
    y_pred_proba = model_cv.predict_proba(X_val_cv)[:, 1]
    
    # Store results
    cv_scores['roc_auc'].append(roc_auc_score(y_val_cv, y_pred_proba))
    cv_scores['accuracy'].append(accuracy_score(y_val_cv, y_pred))
    cv_scores['precision'].append(precision_score(y_val_cv, y_pred))
    cv_scores['recall'].append(recall_score(y_val_cv, y_pred))
    cv_scores['f1'].append(f1_score(y_val_cv, y_pred))
    cv_scores['brier'].append(brier_score_loss(y_val_cv, y_pred_proba))
    
    # Store for calibration - append as numpy arrays
    y_true_cv = np.append(y_true_cv, y_val_cv.values)
    y_pred_proba_cv = np.append(y_pred_proba_cv, y_pred_proba)
    
    print(f"  Fold {fold}: AUC={cv_scores['roc_auc'][-1]:.4f}, "
          f"F1={cv_scores['f1'][-1]:.4f}, "
          f"Brier={cv_scores['brier'][-1]:.4f}")

# Verify arrays have same length
assert len(y_true_cv) == len(y_pred_proba_cv), \
    f"Length mismatch: y_true_cv={len(y_true_cv)}, y_pred_proba_cv={len(y_pred_proba_cv)}"

# Calculate mean and std of CV scores
cv_results = pd.DataFrame({
    'Metric': ['AUC-ROC', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'Brier Score'],
    'Mean': [np.mean(cv_scores['roc_auc']),
             np.mean(cv_scores['accuracy']),
             np.mean(cv_scores['precision']),
             np.mean(cv_scores['recall']),
             np.mean(cv_scores['f1']),
             np.mean(cv_scores['brier'])],
    'Std': [np.std(cv_scores['roc_auc']),
            np.std(cv_scores['accuracy']),
            np.std(cv_scores['precision']),
            np.std(cv_scores['recall']),
            np.std(cv_scores['f1']),
            np.std(cv_scores['brier'])],
    'Min': [np.min(cv_scores['roc_auc']),
            np.min(cv_scores['accuracy']),
            np.min(cv_scores['precision']),
            np.min(cv_scores['recall']),
            np.min(cv_scores['f1']),
            np.min(cv_scores['brier'])],
    'Max': [np.max(cv_scores['roc_auc']),
            np.max(cv_scores['accuracy']),
            np.max(cv_scores['precision']),
            np.max(cv_scores['recall']),
            np.max(cv_scores['f1']),
            np.max(cv_scores['brier'])]
            Brier Score': f"{metrics_ci['brier']['value']:.4f}",
            'CV AUC (mean ± std)': f"{cv_auc_mean:.4f} ± {cv_auc_std:.4f}",
            'Best Params': str(grid_search.best_params_)
        })

print("\nCross-Validation Results:")
print("-"*40)
for _, row in cv_results.iterrows():
    print(f"{row['Metric']:15s}: Mean={row['Mean']:.4f}, Std={row['Std']:.4f}, "
          f"Min={row['Min']:.4f}, Max={row['Max']:.4f}")
# Save CV results to CSV
cv_results.to_csv('cross_validation_results.csv', index=False)
print("\n✅ Cross-validation results saved to 'cross_validation_results.csv'")

# Check for test sets in different naming conventions
test_data_exists = False
X_test_data = None
y_test_data = None

if 'X_test_rf' in locals() and 'y_test_rf' in locals():
    X_test_data = X_test_rf
    y_test_data = y_test_rf
    test_data_exists = True
elif 'X_test' in locals() and 'y_test' in locals():
    X_test_data = X_test
    y_test_data = y_test
    test_data_exists = True

if test_data_exists:
    # Predict on test set
    y_test_pred = stroke_model.predict(X_test_data)
    y_test_pred_proba = stroke_model.predict_proba(X_test_data)[:, 1]
    
    # Calculate metrics
    test_metrics = {
        'AUC-ROC': roc_auc_score(y_test_data, y_test_pred_proba),
        'Accuracy': accuracy_score(y_test_data, y_test_pred),
        'Precision': precision_score(y_test_data, y_test_pred),
        'Recall': recall_score(y_test_data, y_test_pred),
        'F1-Score': f1_score(y_test_data, y_test_pred),
        'Brier Score': brier_score_loss(y_test_data, y_test_pred_proba)
    }
    
    print("\nTest Set Performance:")
    print("-"*40)
    for metric, value in test_metrics.items():
        print(f"{metric:15s}: {value:.4f}")
    
    # Confusion matrix
    cm = confusion_matrix(y_test_data, y_test_pred)
    cm_df = pd.DataFrame(cm, 
                         index=['No Stroke', 'Stroke'],
                         columns=['Predicted No Stroke', 'Predicted Stroke'])
    print("\nConfusion Matrix:")
    print(cm_df)
    
    # Classification report
    print("\nClassification Report:")
    print(classification_report(y_test_data, y_test_pred, 
                                target_names=['No Stroke', 'Stroke']))
else:
    print("No external test set found for validation.")
    # Classification report
    print("\nClassification Report:")
    print(classification_report(y_test_data, y_test_pred, 
                                target_names=['No Stroke', 'Stroke']))


# ====== CALIBRATION ANALYSIS ======
print("\n" + "="*80)
print("CALIBRATION METRICS")
print("="*80)

# Calculate Brier scores
if test_data_exists:
    brier_train = brier_score_loss(y_train_full, cross_val_predict(stroke_model, X_train_full, y_train_full, 
                                                                   cv=cv, method='predict_proba')[:, 1])
    brier_test = brier_score_loss(y_test_data, y_test_pred_proba)
    
    print(f"\nBrier Scores:")
    print(f"  Training (CV): {brier_train:.4f}")
    print(f"  Test Set:      {brier_test:.4f}")
    
    # Calibration curves
    prob_true_cv, prob_pred_cv = calibration_curve(y_true_cv, y_pred_proba_cv, n_bins=10, strategy='uniform')
    prob_true_test, prob_pred_test = calibration_curve(y_test_data, y_test_pred_proba, n_bins=10, strategy='uniform')
    
    # Perfect calibration line
    perfect_calibration = np.linspace(0, 1, 100)
    
    # Plot calibration curves
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Cross-validation calibration
    ax1.plot(prob_pred_cv, prob_true_cv, 's-', label='Model Calibration', linewidth=2, markersize=8)
    ax1.plot(perfect_calibration, perfect_calibration, 'k--', label='Perfect Calibration', linewidth=2)
    ax1.set_xlabel('Mean Predicted Probability', fontsize=12)
    ax1.set_ylabel('Fraction of Positives', fontsize=12)
    ax1.set_title('Calibration Curve - Cross-Validation', fontsize=14, fontweight='bold')
    ax1.legend(loc='best')
    ax1.grid(True, alpha=0.3)
    
    # Test set calibration
    ax2.plot(prob_pred_test, prob_true_test, 's-', label='Model Calibration', linewidth=2, markersize=8)
    ax2.plot(perfect_calibration, perfect_calibration, 'k--', label='Perfect Calibration', linewidth=2)
    ax2.set_xlabel('Mean Predicted Probability', fontsize=12)
    ax2.set_ylabel('Fraction of Positives', fontsize=12)
    ax2.set_title('Calibration Curve - Test Set', fontsize=14, fontweight='bold')
    ax2.legend(loc='best')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('calibration_curves.png', dpi=300, bbox_inches='tight')
    print("\n✅ Calibration curves saved to 'calibration_curves.png'")
    
    # Calibration statistics
    calibration_df = pd.DataFrame({
        'Dataset': ['Cross-Validation', 'Test Set'],
        'Brier Score': [brier_train, brier_test],
        'Calibration Slope': [
            np.polyfit(prob_pred_cv, prob_true_cv, 1)[0],
            np.polyfit(prob_pred_test, prob_true_test, 1)[0]
        ],
        'Calibration Intercept': [
            np.polyfit(prob_pred_cv, prob_true_cv, 1)[1],
            np.polyfit(prob_pred_test, prob_true_test, 1)[1]
        ]
    })
    
    print("\nCalibration Statistics:")
    print(calibration_df.to_string(index=False))
else:
    print("No test set available for calibration analysis.")

# ====== DECISION CURVE ANALYSIS ======
print("\n" + "="*80)
print("DECISION CURVE ANALYSIS")
print("="*80)

def calculate_net_benefit(y_true, y_pred_proba, threshold):
    """
    Calculate net benefit for a given threshold.
    Based on: Vickers & Elkin (2006) Decision Curve Analysis
    """
    # Convert probabilities to binary predictions at threshold
    y_pred = (y_pred_proba >= threshold).astype(int)
    
    # Calculate confusion matrix components
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    n = len(y_true)
    
    # Calculate net benefit
    if threshold == 0:
        net_benefit = np.mean(y_true)  # Treat all
    elif threshold == 1:
        net_benefit = 0  # Treat none
    else:
        net_benefit = (tp / n) - (fp / n) * (threshold / (1 - threshold))
    
    return net_benefit

def decision_curve_analysis(y_true, y_pred_proba, model_name="Model"):
    """
    Perform Decision Curve Analysis
    """
    # Define threshold range (0 to 1, including 0 and 1)
    thresholds = np.linspace(0, 1, 101)
    
    # Calculate net benefit for model
    net_benefits = []
    for threshold in thresholds:
        net_benefits.append(calculate_net_benefit(y_true, y_pred_proba, threshold))
    
    # Calculate net benefit for treat-all and treat-none strategies
    prevalence = np.mean(y_true)
    treat_all_nb = [prevalence for _ in thresholds]
    treat_none_nb = [0] * len(thresholds)
    
    return thresholds, net_benefits, treat_all_nb, treat_none_nb

# Perform DCA for test set if available
if test_data_exists:
    thresholds, model_nb, treat_all_nb, treat_none_nb = decision_curve_analysis(
        y_test_data, y_test_pred_proba, "Stroke Model"
    )
    
    # Plot Decision Curve
    plt.figure(figsize=(12, 8))
    plt.plot(thresholds, model_nb, 'b-', linewidth=3, label='Stroke Model')
    plt.plot(thresholds, treat_all_nb, 'r--', linewidth=2, label='Treat All')
    plt.plot(thresholds, treat_none_nb, 'k--', linewidth=2, label='Treat None')
    
    plt.xlabel('Threshold Probability', fontsize=12)
    plt.ylabel('Net Benefit', fontsize=12)
    plt.title('Decision Curve Analysis - Stroke Risk Model', fontsize=14, fontweight='bold')
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)
    plt.ylim([-0.05, max(max(model_nb), max(treat_all_nb)) * 1.1])
    
    plt.tight_layout()
    plt.savefig('decision_curve_analysis.png', dpi=300, bbox_inches='tight')
    print("\n✅ Decision curve analysis saved to 'decision_curve_analysis.png'")
    
    # Calculate clinical impact at different thresholds
    print("\nClinical Impact at Different Risk Thresholds:")
    print("-"*60)
    print(f"{'Threshold':>12} {'Net Benefit':>15} {'TP per 1000':>12} {'FP per 1000':>12}")
    print("-"*60)
    
    clinical_thresholds = [0.05, 0.1, 0.2, 0.3, 0.5]
    for t in clinical_thresholds:
        nb = calculate_net_benefit(y_test_data, y_test_pred_proba, t)
        y_pred_t = (y_test_pred_proba >= t).astype(int)
        tp_per_1000 = np.sum((y_test_data == 1) & (y_pred_t == 1)) / len(y_test_data) * 1000
        fp_per_1000 = np.sum((y_test_data == 0) & (y_pred_t == 1)) / len(y_test_data) * 1000
        
        print(f"{t:>12.2f} {nb:>15.4f} {tp_per_1000:>12.1f} {fp_per_1000:>12.1f}")
    
    # Find optimal threshold (max net benefit)
    optimal_idx = np.argmax(model_nb)
    optimal_threshold = thresholds[optimal_idx]
    optimal_nb = model_nb[optimal_idx]
    
    print(f"\nOptimal Threshold: {optimal_threshold:.3f}")
    print(f"Maximum Net Benefit: {optimal_nb:.4f}")
else:
    print("No test set available for decision curve analysis.")

# ====== COMPREHENSIVE MODEL REPORT ======
print("\n" + "="*80)
print("COMPREHENSIVE MODEL PERFORMANCE REPORT")
print("="*80)

# Create performance summary
performance_summary = {}

# Cross-validation metrics
performance_summary['CV_AUC_mean'] = np.mean(cv_scores['roc_auc'])
performance_summary['CV_AUC_std'] = np.std(cv_scores['roc_auc'])
performance_summary['CV_F1_mean'] = np.mean(cv_scores['f1'])
performance_summary['CV_F1_std'] = np.std(cv_scores['f1'])
performance_summary['CV_Brier_mean'] = np.mean(cv_scores['brier'])

# Test set metrics (if available)
if test_data_exists:
    performance_summary['Test_AUC'] = test_metrics['AUC-ROC']
    performance_summary['Test_F1'] = test_metrics['F1-Score']
    performance_summary['Test_Brier'] = test_metrics['Brier Score']
    performance_summary['Test_Accuracy'] = test_metrics['Accuracy']
    performance_summary['Test_Precision'] = test_metrics['Precision']
    performance_summary['Test_Recall'] = test_metrics['Recall']

# Calibration metrics
if test_data_exists and 'prob_true_test' in locals():
    performance_summary['Calibration_Slope'] = np.polyfit(prob_pred_test, prob_true_test, 1)[0]
    performance_summary['Calibration_Intercept'] = np.polyfit(prob_pred_test, prob_true_test, 1)[1]

# Convert to DataFrame for nice display
performance_df = pd.DataFrame.from_dict(performance_summary, orient='index', columns=['Value'])
print("\nModel Performance Summary:")
print("-"*40)
for idx, row in performance_df.iterrows():
    print(f"{idx:25s}: {row['Value']:.4f}")

# Save performance metrics
performance_df.to_csv('model_performance_metrics.csv')
print("\n✅ Performance metrics saved to 'model_performance_metrics.csv'")

# ====== SHAP ANALYSIS (for feature importance) ======
print("\n" + "="*80)
print("FEATURE IMPORTANCE ANALYSIS (SHAP)")
print("="*80)

# Calculate SHAP values
print("\nCalculating SHAP values for model interpretation...")
explainer = shap.TreeExplainer(stroke_model)

# Determine sample size
sample_size = min(100, len(X_test_data) if test_data_exists else len(X_train_full))
if test_data_exists:
    X_sample = X_test_data.iloc[:sample_size]
else:
    X_sample = X_train_full.iloc[:sample_size]

shap_values = explainer.shap_values(X_sample)

# Handle SHAP output
if isinstance(shap_values, list) and len(shap_values) == 2:
    shap_values_stroke = shap_values[1]
elif isinstance(shap_values, np.ndarray) and len(shap_values.shape) == 3:
    shap_values_stroke = shap_values[:, :, 1]
else:
    shap_values_stroke = shap_values

# Create SHAP summary
mean_abs_shap = np.abs(shap_values_stroke).mean(axis=0)
feature_names = X_sample.columns.tolist()

shap_df = pd.DataFrame({
    'Feature': feature_names,
    'Mean_Absolute_SHAP': mean_abs_shap,
    'Mean_SHAP': shap_values_stroke.mean(axis=0)
}).sort_values('Mean_Absolute_SHAP', ascending=False)

shap_df['Impact'] = shap_df['Mean_SHAP'].apply(
    lambda x: 'Increases Risk' if x > 0 else 'Decreases Risk'
)

print("\nTop 10 Most Important Features:")
print(shap_df.head(10).to_string(index=False))

# Save SHAP results
shap_df.to_csv('shap_feature_importance.csv', index=False)
print("\n✅ SHAP feature importance saved to 'shap_feature_importance.csv'")

# ====== RECOMMENDATIONS AND INSIGHTS ======
print("\n" + "="*80)
print("CLINICAL AND PRACTICAL RECOMMENDATIONS")
print("="*80)

print("\n1. Model Performance Assessment:")
print("   • Cross-validation shows consistent performance across folds")
print("   • Calibration metrics indicate how well predicted probabilities match actual outcomes")
print("   • Decision curve analysis helps determine clinical utility")

print("\n2. Key Performance Indicators:")
print("   • AUC > 0.7: Acceptable discrimination")
print("   • AUC > 0.8: Good discrimination")
print("   • AUC > 0.9: Excellent discrimination")
print("   • Brier score < 0.25: Good calibration")
print("   • Brier score < 0.1: Excellent calibration")

print("\n3. Clinical Implementation Guidelines:")
if test_data_exists:
    print(f"   • Optimal threshold from DCA: {optimal_threshold:.3f}")
    print("   • Use this threshold for clinical decision-making")
print("   • Consider cost-benefit ratio when choosing threshold")
print("   • Monitor calibration over time with new data")
print("   • Validate in external populations before widespread adoption")

print("\n4. Next Steps:")
print("   • External validation in different populations")
print("   • Economic analysis of implementation")
print("   • Development of clinical decision support tool")
print("   • Ongoing monitoring and model updating")

print("\n" + "="*80)
print("VALIDATION AND ANALYSIS COMPLETE")
print("="*80)
print("\nGenerated files:")
print("  • calibration_curves.png - Calibration plots")
print("  • decision_curve_analysis.png - DCA plot")
print("  • model_performance_metrics.csv - All performance metrics")
print("  • shap_feature_importance.csv - Feature importance scores")


# Display results
print("\n" + "="*100)
print("MODEL COMPARISON RESULTS (70% Stroke, 30% Non-Stroke Dataset)")
print("="*100)
print(f"\nDataset size: {len(df)} samples ({sum(y==1)} stroke, {sum(y==0)} non-stroke)")
print(f"Train size: {len(X_train_multi)}, Test size: {len(X_test_multi)}")
print(f"Features analyzed: {len(X.columns)}")
print("\n")
    
# Display formatted table
pd.set_option('display.max_colwidth', None)
# Safely print only columns that exist to avoid KeyError
desired_cols = [
    'Algorithm', 'Accuracy (%)', 'Sensitivity (%)', 'Specificity (%)',
    'PPV (%)', 'NPV (%)', 'AUC', 'Brier Score', 'CV AUC(mean ± std)'
]
cols_to_show = [c for c in desired_cols if c in df_results.columns]
if not cols_to_show:
    print(df_results.to_string(index=False))
else:
    print(df_results[cols_to_show].to_string(index=False))
    


SyntaxError: unterminated string literal (detected at line 1381) (2503495973.py, line 1381)

In [8]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ====== IMPORT ALL NECESSARY LIBRARIES ======
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report, 
    precision_score, recall_score, f1_score, confusion_matrix, 
    roc_curve, auc
)
from sklearn.pipeline import make_pipeline
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import label_binarize
import joblib
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, label_binarize
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    roc_auc_score, confusion_matrix, brier_score_loss, roc_curve, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score, classification_report, auc
)
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, HistGradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.multiclass import OneVsRestClassifier
from sklearn.datasets import make_classification
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ====== LOAD STROKE DATASET ======
df = pd.read_csv(r"C:\Users\sibs2\african-neurohealth-dashboard\brain_stroke_70_percent_stroke.csv")

print(f"Stroke dataset shape: {df.shape}")
print(f"Stroke dataset columns: {list(df.columns)}")

# ====== ENCODE TARGET VARIABLE ======
if 'stroke' in df.columns:
    print(f"Unique stroke values: {df['stroke'].unique()}")
    print(f"Target distribution before encoding: {df['stroke'].value_counts()}")
    y = df['stroke'].astype(int)
else:
    raise ValueError("stroke column not found!")

print(f"\nTarget distribution:")
print(f"Stroke (1): {(y == 1).sum()} ({(y == 1).mean()*100:.1f}%)")
print(f"No Stroke (0): {(y == 0).sum()} ({(y == 0).mean()*100:.1f}%)")

# ====== PREPARE FEATURES ======
X = df.drop('stroke', axis=1)

print(f"\nOriginal feature types:")
for col in X.columns:
    print(f"  {col}: {X[col].dtype}, unique values: {len(X[col].unique())}")

# ====== ENCODE CATEGORICAL VARIABLES ======
print("\n=== Encoding categorical variables ===")
label_encoders = {}
for col in X.select_dtypes(include=['object', 'bool']).columns:
    print(f"Encoding {col}: {sorted(X[col].unique())[:5]}")
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

# ====== HANDLE MISSING VALUES ======
print("\n=== Handling missing values ===")

# Separate numerical and categorical columns
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'bool']).columns.tolist()

print(f"Numeric features ({len(numeric_features)}): {numeric_features[:5] if len(numeric_features) > 5 else numeric_features}...")
print(f"Categorical features ({len(categorical_features)}): {categorical_features[:5] if len(categorical_features) > 5 else categorical_features}...")

# Initialize as empty lists if none exist
if not numeric_features:
    numeric_features = []
if not categorical_features:
    categorical_features = []

# --- 2. Create a Preprocessor using ColumnTransformer ---
# This applies the right imputer to the right column type
numeric_imputer = SimpleImputer(strategy='median')
categorical_imputer = SimpleImputer(strategy='most_frequent')

transformers = []
if len(numeric_features) > 0:
    transformers.append(('num', numeric_imputer, numeric_features))
if len(categorical_features) > 0:
    transformers.append(('cat', categorical_imputer, categorical_features))

if len(transformers) == 0:
    raise ValueError("No features found in dataset")

preprocessor = ColumnTransformer(transformers=transformers)
# Set output to pandas DataFrame to keep column names
preprocessor.set_output(transform="pandas")

# --- 3. Apply Imputation to Features (X) ---
# Fit on the entire feature set to learn values, then transform
print("Fitting imputer and transforming data...")
X_imputed = preprocessor.fit_transform(X)

# Check again to confirm no missing values remain
if X_imputed.isnull().sum().sum() == 0:
    print("✅ Success: All missing values have been imputed.")
else:
    print("⚠️ Warning: Some missing values remain. Check data.")
    print(X_imputed.isnull().sum())

# Replace the original X with the imputed version
X = X_imputed

print(f"\nFinal X shape after preprocessing: {X.shape}")

# ====== SPLIT DATA (FOR BOTH ANALYSES) ======
# First split for Random Forest and SHAP
X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Second split for multi-classifier comparison (70/30 split)
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"\n=== Data Splits ===")
print(f"Random Forest (80/20):")
print(f"  X_train: {X_train_rf.shape}, X_test: {X_test_rf.shape}")
print(f"  y_train distribution: {np.bincount(y_train_rf)}")
print(f"  y_test distribution: {np.bincount(y_test_rf)}")

print(f"\nMulti-Classifier (70/30):")
print(f"  X_train: {X_train_multi.shape}, X_test: {X_test_multi.shape}")
print(f"  y_train distribution: {np.bincount(y_train_multi)}")
print(f"  y_test distribution: {np.bincount(y_test_multi)}")

# ===========================================
# PART 1: RANDOM FOREST WITH SHAP (80/20 SPLIT)
# ===========================================
print("\n" + "="*80)
print("PART 1: RANDOM FOREST WITH SHAP ANALYSIS")
print("="*80)

# ====== TRAIN RANDOM FOREST ======
print("\n=== Training Random Forest ===")
stroke_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)

stroke_model.fit(X_train_rf, y_train_rf)

# ====== EVALUATE ======
y_pred_rf = stroke_model.predict(X_test_rf)
y_proba_rf = stroke_model.predict_proba(X_test_rf)
if y_proba_rf.ndim > 1:
    y_proba_rf = y_proba_rf[:, 1]
else:
    y_proba_rf = np.array(y_proba_rf).flatten()

# ====== SET VISUALIZATION STYLE ======

# Set style for better visualizations
from matplotlib import rcParams

plt.style.use('seaborn-v0_8-whitegrid')
rcParams['figure.figsize'] = (12, 8)
rcParams['font.size'] = 11
sns.set_palette("husl")
joblib.dump(stroke_model, "stroke_REAL_model.pkl")
print("\n✅ STROKE model saved as stroke_REAL_model.pkl")


# ====== SET VISUALIZATION STYLE ======

# Set style for better visualizations
plt.style.use('seaborn-v0_8-whitegrid')
rcParams['figure.figsize'] = (12, 8)
rcParams['font.size'] = 11
sns.set_palette("husl")

# ====== 10-FOLD CROSS-VALIDATION ======
print("\n" + "="*80)
print("10-FOLD CROSS-VALIDATION")
print("="*80)

# Prepare data for cross-validation
# Check which variables exist
if 'X_train_rf' in locals() and 'X_test_rf' in locals():
    X_train_full = pd.concat([X_train_rf, X_test_rf])
    y_train_full = pd.concat([y_train_rf, y_test_rf])
    print(f"Using combined train+test data: {X_train_full.shape[0]} samples, {X_train_full.shape[1]} features")
elif 'X_train' in locals() and 'X_test' in locals():
    X_train_full = pd.concat([X_train, X_test])
    y_train_full = pd.concat([y_train, y_test])
    print(f"Using combined train+test data: {X_train_full.shape[0]} samples, {X_train_full.shape[1]} features")
else:
    X_train_full = X
    y_train_full = y
    print(f"Using full dataset: {X_train_full.shape[0]} samples, {X_train_full.shape[1]} features")

# Display all features
print(f"\nAll {len(X_train_full.columns)} features in dataset:")
for i, feature in enumerate(X_train_full.columns, 1):
    print(f"{i:3d}. {feature}")

# Initialize stratified K-Fold
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Store CV results
cv_scores = {
    'roc_auc': [],
    'accuracy': [],
    'precision': [],
    'recall': [],
    'f1': [],
    'brier': []
}

# Store predictions for calibration
y_true_cv = []
y_pred_proba_cv = []

# Cross-validation loop
print("\nPerforming 10-fold cross-validation...")
for fold, (train_idx, val_idx) in enumerate(cv.split(X_train_full, y_train_full), 1):
    X_train_cv, X_val_cv = X_train_full.iloc[train_idx], X_train_full.iloc[val_idx]
    y_train_cv, y_val_cv = y_train_full.iloc[train_idx], y_train_full.iloc[val_idx]
    
    # Train model on training fold
    model_cv = stroke_model.__class__(**stroke_model.get_params())
    model_cv.fit(X_train_cv, y_train_cv)
    
    # Predict on validation fold
    y_pred = model_cv.predict(X_val_cv)
    y_pred_proba = model_cv.predict_proba(X_val_cv)[:, 1]
    
    # Store results
    cv_scores['roc_auc'].append(roc_auc_score(y_val_cv, y_pred_proba))
    cv_scores['accuracy'].append(accuracy_score(y_val_cv, y_pred))
    cv_scores['precision'].append(precision_score(y_val_cv, y_pred))
    cv_scores['recall'].append(recall_score(y_val_cv, y_pred))
    cv_scores['f1'].append(f1_score(y_val_cv, y_pred))
    cv_scores['brier'].append(brier_score_loss(y_val_cv, y_pred_proba))
    
    # Store for calibration
    y_true_cv.extend(y_val_cv)
    y_pred_proba_cv.extend(y_pred_proba)
    
    print(f"  Fold {fold}: AUC={cv_scores['roc_auc'][-1]:.4f}, "
          f"F1={cv_scores['f1'][-1]:.4f}, "
          f"Brier={cv_scores['brier'][-1]:.4f}")

# Calculate mean and std of CV scores
cv_results = pd.DataFrame({
    'Metric': ['AUC-ROC', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'Brier Score'],
    'Mean': [np.mean(cv_scores['roc_auc']),
             np.mean(cv_scores['accuracy']),
             np.mean(cv_scores['precision']),
             np.mean(cv_scores['recall']),
             np.mean(cv_scores['f1']),
             np.mean(cv_scores['brier'])],
    'Std': [np.std(cv_scores['roc_auc']),
            np.std(cv_scores['accuracy']),
            np.std(cv_scores['precision']),
            np.std(cv_scores['recall']),
            np.std(cv_scores['f1']),
            np.std(cv_scores['brier'])],
    'Min': [np.min(cv_scores['roc_auc']),
            np.min(cv_scores['accuracy']),
            np.min(cv_scores['precision']),
            np.min(cv_scores['recall']),
            np.min(cv_scores['f1']),
            np.min(cv_scores['brier'])],
    'Max': [np.max(cv_scores['roc_auc']),
            np.max(cv_scores['accuracy']),
            np.max(cv_scores['precision']),
            np.max(cv_scores['recall']),
            np.max(cv_scores['f1']),
            np.max(cv_scores['brier'])]
})

print("\n" + "-"*80)
print("CROSS-VALIDATION RESULTS (10-fold)")
print("-"*80)
print(cv_results.to_string(index=False))

# ====== SHAP ANALYSIS - ALL FEATURES ======
print("\n" + "="*80)
print("SHAP ANALYSIS - ALL FEATURES")
print("="*80)

# Create directory for SHAP plots
import os
shap_dir = 'shap_analysis_all_features'
if not os.path.exists(shap_dir):
    os.makedirs(shap_dir)

# Calculate SHAP values for ALL features
print("\nCalculating SHAP values for model interpretation...")
explainer = shap.TreeExplainer(stroke_model)

# Determine which data to use for SHAP
if 'X_test_rf' in locals() and len(X_test_rf) > 0:
    X_shap = X_test_rf
    print(f"Using test set for SHAP: {X_shap.shape[0]} samples, {X_shap.shape[1]} features")
elif 'X_test' in locals() and len(X_test) > 0:
    X_shap = X_test
    print(f"Using test set for SHAP: {X_shap.shape[0]} samples, {X_shap.shape[1]} features")
else:
    X_shap = X_train_full
    print(f"Using training set for SHAP: {X_shap.shape[0]} samples, {X_shap.shape[1]} features")

# Limit sample size for computational efficiency but ensure we get good representation
sample_size = min(200, len(X_shap))
X_sample = X_shap.iloc[:sample_size]
print(f"\nUsing {sample_size} samples for SHAP calculation")
print(f"All {X_sample.shape[1]} features will be analyzed:")

# Display all features being analyzed
for i, feature in enumerate(X_sample.columns, 1):
    print(f"{i:3d}. {feature}")

# Calculate SHAP values
shap_values = explainer.shap_values(X_sample)

# Handle SHAP output
if isinstance(shap_values, list) and len(shap_values) == 2:
    shap_values_stroke = shap_values[1]
    print("Using class 1 SHAP values (Stroke)")
elif isinstance(shap_values, np.ndarray) and len(shap_values.shape) == 3:
    shap_values_stroke = shap_values[:, :, 1]
    print("Using class 1 SHAP values from 3D array")
else:
    shap_values_stroke = shap_values
    print("Using single class SHAP values")

# Get all feature names
feature_names = X_sample.columns.tolist()
print(f"\nAnalyzing {len(feature_names)} features total")

# Calculate SHAP statistics for ALL features
mean_abs_shap = np.abs(shap_values_stroke).mean(axis=0)
mean_shap = shap_values_stroke.mean(axis=0)

# Create comprehensive SHAP DataFrame for ALL features
shap_df = pd.DataFrame({
    'Feature': feature_names,
    'Mean_Absolute_SHAP': mean_abs_shap,
    'Mean_SHAP': mean_shap
}).sort_values('Mean_Absolute_SHAP', ascending=False)

# Determine impact direction for ALL features
shap_df['Impact'] = shap_df['Mean_SHAP'].apply(
    lambda x: 'Increases Risk' if x > 0 else 'Decreases Risk'
)

# Display all features with SHAP values
print("\n" + "="*80)
print("ALL FEATURES - SHAP VALUES (Sorted by Importance)")
print("="*80)
print(f"{'Rank':>15} {'Feature':<30} {'Impact':>15} {'Mean_SHAP':>12} {'Abs_SHAP':>12}")
print("-"*80)

for i, (idx, row) in enumerate(shap_df.iterrows(), 1):
    print(f"{i:>15} {row['Feature']:<30} {row['Impact']:>15} {row['Mean_SHAP']:>12.4f} {row['Mean_Absolute_SHAP']:>12.4f}")

# Save complete SHAP results
shap_df.to_csv(f'{shap_dir}/shap_all_features_complete.csv', index=False)
print(f"\n✅ Complete SHAP values for ALL features saved to '{shap_dir}/shap_all_features_complete.csv'")

# ====== SHAP VISUALIZATIONS - ALL FEATURES ======
print("\n" + "="*80)
print("SHAP VISUALIZATIONS - ALL FEATURES")
print("="*80)

# 1. BAR PLOT - ALL Features (Global Importance)
print("\n1. Generating SHAP Bar Plot - ALL Features...")
plt.figure(figsize=(16, max(20, len(feature_names) * 0.4)))  # Dynamic height based on number of features
shap.summary_plot(shap_values_stroke, X_sample, plot_type="bar", 
                  show=False, max_display=len(feature_names))
plt.title(f"SHAP Feature Importance - ALL {len(feature_names)} Features", 
          fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(f'{shap_dir}/shap_bar_plot_ALL_features.png', dpi=300, bbox_inches='tight')
plt.close()
print(f"   ✅ Saved: {shap_dir}/shap_bar_plot_ALL_features.png")

# 2. BEESWARM PLOT - ALL Features (shows feature values and SHAP)
print("2. Generating SHAP Beeswarm Plot - ALL Features...")
plt.figure(figsize=(18, max(12, len(feature_names) * 0.5)))
shap.summary_plot(shap_values_stroke, X_sample, 
                  show=False, max_display=len(feature_names))
plt.title(f"SHAP Values Impact - ALL {len(feature_names)} Features", 
          fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(f'{shap_dir}/shap_beeswarm_ALL_features.png', dpi=300, bbox_inches='tight')
plt.close()
print(f"   ✅ Saved: {shap_dir}/shap_beeswarm_ALL_features.png")


# 4. CUSTOM SORTED BAR PLOT - All Features with Colors
print("4. Generating Custom Sorted SHAP Bar Plot - ALL Features...")
fig, ax = plt.subplots(figsize=(14, max(10, len(feature_names) * 0.4)))

# Sort features by absolute SHAP value
sorted_idx = np.argsort(np.abs(shap_values_stroke).mean(0))[::-1]
sorted_features = [feature_names[i] for i in sorted_idx]
sorted_mean_shap = [mean_shap[i] for i in sorted_idx]

# Color by positive/negative impact
colors = ['#FF6B6B' if val > 0 else '#4ECDC4' for val in sorted_mean_shap]  # Red for risk, green for protective

y_pos = np.arange(len(sorted_features))
bars = ax.barh(y_pos, sorted_mean_shap, align='center', color=colors, alpha=0.8, height=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(sorted_features, fontsize=10)
ax.invert_yaxis()
ax.set_xlabel('Mean SHAP Value (Impact on Stroke Risk)', fontsize=12)
ax.set_title(f'ALL {len(feature_names)} Features: SHAP Value Impact\n(Red = Increases Risk, Green = Decreases Risk)', 
             fontsize=14, fontweight='bold')

# Add value labels
for i, (bar, v) in enumerate(zip(bars, sorted_mean_shap)):
    width = bar.get_width()
    ax.text(width + (0.01 if width >= 0 else -0.01), bar.get_y() + bar.get_height()/2,
            f'{v:.4f}', 
            ha='left' if width >= 0 else 'right',
            va='center',
            fontsize=9,
            color='black' if abs(width) > 0.01 else 'gray')

plt.tight_layout()
plt.savefig(f'{shap_dir}/shap_custom_bar_ALL_features.png', dpi=300, bbox_inches='tight')
plt.close()
print(f"   ✅ Saved: {shap_dir}/shap_custom_bar_ALL_features.png")

# 5. CATEGORICAL ANALYSIS - Group features by type
print("\n5. Categorizing features by type...")
clinical_keywords = ['age', 'blood', 'pressure', 'glucose', 'sugar', 'diabet', 'heart', 'bmi', 
                     'weight', 'height', 'smok', 'alcohol', 'cholest', 'lipid', 'stress', 
                     'sleep', 'depress', 'ptsd', 'anxiety', 'stroke', 'attack', 'failure']

demographic_keywords = ['gender', 'sex', 'age', 'race', 'ethnic', 'marital', 'marriage', 
                        'education', 'employ', 'work', 'income', 'region', 'urban', 'rural']

lifestyle_keywords = ['smok', 'alcohol', 'drug', 'exercise', 'physical', 'activity', 'diet', 
                      'food', 'nutrition', 'sleep', 'stress']

biometric_keywords = ['bmi', 'weight', 'height', 'waist', 'hip', 'ratio', 'pressure', 
                      'pulse', 'rate', 'respir', 'oxygen', 'temp']

# Categorize each feature
feature_categories = {}
for feature in feature_names:
    feature_lower = feature.lower()
    category = 'Other/Unknown'
    
    if any(keyword in feature_lower for keyword in clinical_keywords):
        category = 'Clinical/Biomedical'
    elif any(keyword in feature_lower for keyword in demographic_keywords):
        category = 'Demographic'
    elif any(keyword in feature_lower for keyword in lifestyle_keywords):
        category = 'Lifestyle'
    elif any(keyword in feature_lower for keyword in biometric_keywords):
        category = 'Biometric/Vitals'
    
    feature_categories[feature] = category

# Add category to SHAP dataframe
shap_df['Category'] = shap_df['Feature'].map(feature_categories)

# Save categorized SHAP results
shap_df.to_csv(f'{shap_dir}/shap_categorized_features.csv', index=False)
print(f"   ✅ Categorized features saved to '{shap_dir}/shap_categorized_features.csv'")

# Display category summary
print("\nFeature Category Summary:")
print("-"*50)
category_summary = shap_df.groupby('Category').agg(
    Count=('Feature', 'count'),
    Mean_SHAP=('Mean_SHAP', 'mean'),
    Mean_Abs_SHAP=('Mean_Absolute_SHAP', 'mean')
).sort_values('Mean_Abs_SHAP', ascending=False)

print(category_summary.to_string())

# 6. CATEGORY-WISE SHAP PLOTS
print("\n6. Generating category-wise SHAP plots...")
for category in category_summary.index:
    category_features = shap_df[shap_df['Category'] == category]
    if len(category_features) > 0:
        plt.figure(figsize=(12, max(6, len(category_features) * 0.3)))
        
        # Get indices for this category
        cat_indices = [i for i, feat in enumerate(feature_names) if feat in category_features['Feature'].values]
        cat_shap_values = shap_values_stroke[:, cat_indices]
        cat_feature_names = [feature_names[i] for i in cat_indices]
        
        shap.summary_plot(cat_shap_values, X_sample.iloc[:, cat_indices], 
                         feature_names=cat_feature_names,
                         show=False, max_display=len(cat_feature_names),
                         plot_type="dot")
        
        plt.title(f"SHAP Values - {category} Features ({len(cat_feature_names)} features)", 
                  fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'{shap_dir}/shap_{category.replace("/", "_").replace(" ", "_")}.png', 
                    dpi=300, bbox_inches='tight')
        plt.close()
        print(f"   ✅ {category}: {len(category_features)} features")

# ====== EXTERNAL VALIDATION (if test set exists) ======
print("\n" + "="*80)
print("EXTERNAL VALIDATION")
print("="*80)

# Check for test sets in different naming conventions
test_data_exists = False
X_test_data = None
y_test_data = None

if 'X_test_rf' in locals() and 'y_test_rf' in locals():
    X_test_data = X_test_rf
    y_test_data = y_test_rf
    test_data_exists = True
elif 'X_test' in locals() and 'y_test' in locals():
    X_test_data = X_test
    y_test_data = y_test
    test_data_exists = True

if test_data_exists:
    # Predict on test set
    y_test_pred = stroke_model.predict(X_test_data)
    y_test_pred_proba = stroke_model.predict_proba(X_test_data)[:, 1]
    
    # Calculate metrics
    test_metrics = {
        'AUC-ROC': roc_auc_score(y_test_data, y_test_pred_proba),
        'Accuracy': accuracy_score(y_test_data, y_test_pred),
        'Precision': precision_score(y_test_data, y_test_pred),
        'Recall': recall_score(y_test_data, y_test_pred),
        'F1-Score': f1_score(y_test_data, y_test_pred),
        'Brier Score': brier_score_loss(y_test_data, y_test_pred_proba)
    }
    
    print("\nTest Set Performance:")
    print("-"*40)
    for metric, value in test_metrics.items():
        print(f"{metric:15s}: {value:.4f}")
    
    # Confusion matrix
    cm = confusion_matrix(y_test_data, y_test_pred)
    cm_df = pd.DataFrame(cm, 
                         index=['Actual Negative', 'Actual Positive'],
                         columns=['Predicted Negative', 'Predicted Positive'])
    
    print("\nConfusion Matrix:")
    print(cm_df)
    
    # Classification report
    print("\nClassification Report:")
    print(classification_report(y_test_data, y_test_pred, 
                                target_names=['No Stroke', 'Stroke']))
else:
    print("No external test set found for validation.")

# ====== CALIBRATION ANALYSIS ======
if test_data_exists:
    print("\n" + "="*80)
    print("CALIBRATION METRICS")
    print("="*80)
    
    # Calculate Brier scores
    brier_train = brier_score_loss(y_train_full, cross_val_predict(stroke_model, X_train_full, y_train_full, 
                                                                   cv=cv, method='predict_proba')[:, 1])
    brier_test = brier_score_loss(y_test_data, y_test_pred_proba)
    
    print(f"\nBrier Scores:")
    print(f"  Training (CV): {brier_train:.4f}")
    print(f"  Test Set:      {brier_test:.4f}")
    
    # Calibration curves
    prob_true_cv, prob_pred_cv = calibration_curve(y_true_cv, y_pred_proba_cv, n_bins=10, strategy='uniform')
    prob_true_test, prob_pred_test = calibration_curve(y_test_data, y_test_pred_proba, n_bins=10, strategy='uniform')
    
    # Plot calibration curves
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Cross-validation calibration
    ax1.plot(prob_pred_cv, prob_true_cv, 's-', label='Model Calibration', linewidth=2, markersize=8)
    ax1.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration', linewidth=2)
    ax1.set_xlabel('Mean Predicted Probability', fontsize=12)
    ax1.set_ylabel('Fraction of Positives', fontsize=12)
    ax1.set_title('Calibration Curve - Cross-Validation', fontsize=14, fontweight='bold')
    ax1.legend(loc='best')
    ax1.grid(True, alpha=0.3)
    
    # Test set calibration
    ax2.plot(prob_pred_test, prob_true_test, 's-', label='Model Calibration', linewidth=2, markersize=8)
    ax2.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration', linewidth=2)
    ax2.set_xlabel('Mean Predicted Probability', fontsize=12)
    ax2.set_ylabel('Fraction of Positives', fontsize=12)
    ax2.set_title('Calibration Curve - Test Set', fontsize=14, fontweight='bold')
    ax2.legend(loc='best')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{shap_dir}/calibration_curves.png', dpi=300, bbox_inches='tight')
    plt.close()
    print(f"\n✅ Calibration curves saved to '{shap_dir}/calibration_curves.png'")

# ====== COMPREHENSIVE FEATURE ANALYSIS REPORT ======
print("\n" + "="*80)
print("COMPREHENSIVE FEATURE ANALYSIS REPORT")
print("="*80)

# Create performance summary
performance_summary = {}

# Cross-validation metrics
performance_summary['CV_AUC_mean'] = np.mean(cv_scores['roc_auc'])
performance_summary['CV_AUC_std'] = np.std(cv_scores['roc_auc'])
performance_summary['CV_F1_mean'] = np.mean(cv_scores['f1'])
performance_summary['CV_F1_std'] = np.std(cv_scores['f1'])
performance_summary['CV_Brier_mean'] = np.mean(cv_scores['brier'])

# Test set metrics (if available)
if test_data_exists:
    performance_summary['Test_AUC'] = test_metrics['AUC-ROC']
    performance_summary['Test_F1'] = test_metrics['F1-Score']
    performance_summary['Test_Brier'] = test_metrics['Brier Score']

# SHAP summary
performance_summary['Total_Features'] = len(feature_names)
performance_summary['Clinical_Features'] = len(shap_df[shap_df['Category'] == 'Clinical/Biomedical'])
performance_summary['Demographic_Features'] = len(shap_df[shap_df['Category'] == 'Demographic'])
performance_summary['Lifestyle_Features'] = len(shap_df[shap_df['Category'] == 'Lifestyle'])
performance_summary['Top_Feature'] = shap_df.iloc[0]['Feature']
performance_summary['Top_Feature_SHAP'] = shap_df.iloc[0]['Mean_Absolute_SHAP']

# Convert to DataFrame for nice display
performance_df = pd.DataFrame.from_dict(performance_summary, orient='index', columns=['Value'])
print("\nModel Performance Summary:")
print("-"*40)
for idx, row in performance_df.iterrows():
    print(f"{idx:25s}: {row['Value']:.4f}" if isinstance(row['Value'], float) else f"{idx:25s}: {row['Value']}")

# Save performance metrics
performance_df.to_csv(f'{shap_dir}/model_performance_metrics.csv')
print(f"\n✅ Performance metrics saved to '{shap_dir}/model_performance_metrics.csv'")

# ====== FEATURE INSIGHTS SUMMARY ======
print("\n" + "="*80)
print("FEATURE INSIGHTS SUMMARY")
print("="*80)

print(f"\nTotal Features Analyzed: {len(feature_names)}")
print(f"Features by Category:")
for category, count in shap_df['Category'].value_counts().items():
    print(f"  • {category}: {count} features")

print(f"\nTop 5 Features Increasing Stroke Risk:")
top_risk = shap_df[shap_df['Impact'] == 'Increases Risk'].head(5)
for i, (idx, row) in enumerate(top_risk.iterrows(), 1):
    print(f"  {i}. {row['Feature']} (SHAP: {row['Mean_SHAP']:.4f}, Category: {row['Category']})")

print(f"\nTop 5 Features Decreasing Stroke Risk (Protective):")
top_protective = shap_df[shap_df['Impact'] == 'Decreases Risk'].head(5)
for i, (idx, row) in enumerate(top_protective.iterrows(), 1):
    print(f"  {i}. {row['Feature']} (SHAP: {row['Mean_SHAP']:.4f}, Category: {row['Category']})")

print(f"\nFeatures with Negligible Impact (|SHAP| < 0.001):")
negligible = shap_df[shap_df['Mean_Absolute_SHAP'] < 0.001]
print(f"  Count: {len(negligible)} features")
if len(negligible) > 0:
    print(f"  Examples: {', '.join(negligible['Feature'].head(5).tolist())}")

# ====== RECOMMENDATIONS ======
print("\n" + "="*80)
print("RECOMMENDATIONS FOR MODEL DEPLOYMENT")
print("="*80)

print("\n1. Feature Importance Insights:")
print("   • Focus on top risk factors for intervention strategies")
print("   • Consider protective factors for preventive measures")
print("   • Low-impact features might be candidates for feature reduction")

print("\n2. Model Performance:")
print(f"   • Cross-validation AUC: {performance_summary['CV_AUC_mean']:.3f} ± {performance_summary['CV_AUC_std']:.3f}")
if test_data_exists:
    print(f"   • Test set AUC: {performance_summary['Test_AUC']:.3f}")

print("\n3. Clinical Implementation:")
print("   • Consider feature categories when designing interventions")
print("   • Demographic and lifestyle factors may be modifiable")
print("   • Clinical/biomedical factors may require medical intervention")

print("\n4. Next Steps:")
print("   • Validate in external populations")
print("   • Consider feature engineering based on insights")
print("   • Develop targeted interventions for top risk factors")

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)
print(f"\nAll outputs saved to '{shap_dir}/' directory:")
print("  • shap_all_features_complete.csv - Complete SHAP values for ALL features")
print("  • shap_categorized_features.csv - Features categorized by type")
print("  • shap_bar_plot_ALL_features.png - Bar plot with ALL features")
print("  • shap_beeswarm_ALL_features.png - Beeswarm plot with ALL features")
print("  • shap_custom_bar_ALL_features.png - Custom bar plot with ALL features")
print("  • Category-specific SHAP plots")
print("  • model_performance_metrics.csv - Performance summary")
if test_data_exists:
    print("  • calibration_curves.png - Calibration analysis")


# Generate SHAP plot
print("\nGenerating SHAP summary plot...")
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values_stroke, X_sample, feature_names=feature_names, show=False)
plt.tight_layout()
plt.savefig('CORRECTED_stroke_shap_summary.png', dpi=300, bbox_inches='tight')
plt.close()
print("SHAP plot saved to: CORRECTED_stroke_shap_summary.png")

Stroke dataset shape: (3750, 47)
Stroke dataset columns: ['gender', 'age', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'Residence_type', 'avg_glucose_level', 'bmi', 'smoking_status', 'stroke', 'stress_level', 'ptsd', 'depression_level', 'diabetes_type', 'sleep_hours', 'chronic_pain_None', 'chronic_pain_Osteoarthritis', 'chronic_pain_Others', 'chronic_pain_Rheumatism', 'salt_intake_High', 'salt_intake_Little', 'salt_intake_Moderate', 'salt_intake_None', 'hypertension_treatment_Drugs', 'hypertension_treatment_Herbal', 'hypertension_treatment_None', 'nutritional_lifestyle_Fast Foods', 'nutritional_lifestyle_Homemade Food', 'nutritional_lifestyle_Junk Food', 'nutritional_lifestyle_Local Bukka/Street Food', 'noise_sources_Block-Industry', 'noise_sources_Church', 'noise_sources_Club-House', 'noise_sources_Generator', 'noise_sources_Grinding-Machine', 'noise_sources_Market', 'noise_sources_Mosque', 'noise_sources_None', 'noise_sources_Welder', 'pollution_level Air', 'polluti